In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/insurance/insurance.csv


In [2]:
data = pd.read_csv('/kaggle/input/insurance/insurance.csv')
data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
data.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [4]:
data.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='object')

In [5]:
data.drop(['region'], axis=1)

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [6]:
from sklearn.preprocessing import OneHotEncoder
data_encoded = pd.get_dummies(data, columns=['sex', 'smoker'], drop_first=True)
print("\nOne-hot encoded dataset:\n", data_encoded)


One-hot encoded dataset:
       age     bmi  children     region      charges  sex_male  smoker_yes
0      19  27.900         0  southwest  16884.92400     False        True
1      18  33.770         1  southeast   1725.55230      True       False
2      28  33.000         3  southeast   4449.46200      True       False
3      33  22.705         0  northwest  21984.47061      True       False
4      32  28.880         0  northwest   3866.85520      True       False
...   ...     ...       ...        ...          ...       ...         ...
1333   50  30.970         3  northwest  10600.54830      True       False
1334   18  31.920         0  northeast   2205.98080     False       False
1335   18  36.850         0  southeast   1629.83350     False       False
1336   21  25.800         0  southwest   2007.94500     False       False
1337   61  29.070         0  northwest  29141.36030     False        True

[1338 rows x 7 columns]


In [7]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# numerical features to have a mean of 0 and a standard deviation of 1.

data_encoded[['bmi', 'charges']] = scaler.fit_transform(data_encoded[['bmi', 'charges']])
print("\nDataset after feature scaling:\n", data_encoded)


Dataset after feature scaling:
       age       bmi  children     region   charges  sex_male  smoker_yes
0      19 -0.453320         0  southwest  0.298584     False        True
1      18  0.509621         1  southeast -0.953689      True       False
2      28  0.383307         3  southeast -0.728675      True       False
3      33 -1.305531         0  northwest  0.719843      True       False
4      32 -0.292556         0  northwest -0.776802      True       False
...   ...       ...       ...        ...       ...       ...         ...
1333   50  0.050297         3  northwest -0.220551      True       False
1334   18  0.206139         0  northeast -0.914002     False       False
1335   18  1.014878         0  southeast -0.961596     False       False
1336   21 -0.797813         0  southwest -0.930362     False       False
1337   61 -0.261388         0  northwest  1.311053     False        True

[1338 rows x 7 columns]


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X = data_encoded.drop(['region', 'charges'], axis=1)
y = data_encoded['charges']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/4, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Mean Squared Error:", mse)
print("R-squared:", r2)

Coefficients: [0.02148082 0.16494799 0.03591796 0.00545784 1.95186217]
Intercept: -1.2826500216544594
Mean Squared Error: 0.24175928196498894
R-squared: 0.7652077247609824
